# TP2 NLP - Executor Colab

Este notebook apenas orquestra comandos do repositório. A implementação fica em `src`, `custom_metrics` e `scripts`.


## 1. Clonar ou entrar no repositório

Esta versão usa o repositório pessoal público. Se o runtime já tiver uma cópia antiga com conflito de Git, remova a pasta `/content/TP2_NLP` e execute a célula de clone novamente.


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mh131105/TP2_NLP.git'
REPO_DIR = Path('/content/TP2_NLP')
BRANCH = 'main'

if not (REPO_DIR / '.git').exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f'Repositório já existe em {REPO_DIR}')

%cd {REPO_DIR}
!git remote set-url origin {REPO_URL}
!git fetch origin {BRANCH}
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}
!git status --short --branch


In [ ]:
# Use esta célula apenas se o runtime ficou com uma cópia quebrada ou divergente.
# Depois execute novamente a célula de clone acima.
# !rm -rf /content/TP2_NLP


## 2. Instalar dependências


In [ ]:
!grep torch requirements.txt || echo "torch não está fixado no requirements"
!pip install -r requirements.txt


## 3. Login opcional no Hugging Face

Defina `HF_TOKEN` nos segredos do Colab ou deixe a célula seguir sem login para modelos/datasets públicos.


In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    hf_token = hf_token or userdata.get('HF_TOKEN')
except Exception:
    pass

if hf_token:
    login(token=hf_token)
else:
    print('HF_TOKEN não configurado; seguindo sem login.')


## 4. Preparar datasets

O `prepare_spider` primeiro usa `data/raw/spider` se a pasta já existir. Se ela não existir, o script pode importar um diretório ou ZIP/TAR passado com `--source_path`, ou tentar um dataset Hugging Face configurado no script.


In [ ]:
!python -m scripts.prepare_spider --data_dir data/raw/spider --output_dir data/processed/spider
# Se a fonte automática falhar, coloque um ZIP/pasta do Spider no Drive e use:
# !python -m scripts.prepare_spider --data_dir data/raw/spider --output_dir data/processed/spider --source_path /content/drive/MyDrive/spider.zip --force_download
!python -m scripts.prepare_mmlu --config configs/eval.yaml


## 5. Conferir configs disponíveis


In [ ]:
!ls -1 configs/train_lora_exp_*.yaml configs/train_qlora_t4_template.yaml configs/eval.yaml configs/eval_spider_nostop.yaml


## 6. Benchmark do baseline

Roda Spider dev e MMLU 150 para o modelo base, salvando os artefatos em `outputs/base`.


In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/base


## 7. Treinar experimentos LoRA A-H

Cada treino longo fica em uma célula separada para facilitar acompanhamento e retomada no Colab.


In [ ]:
!python -m scripts.train --config configs/train_lora_exp_a.yaml


In [ ]:
!python -m scripts.train --config configs/train_lora_exp_b.yaml


In [ ]:
!python -m scripts.train --config configs/train_lora_exp_c.yaml


In [ ]:
!python -m scripts.train --config configs/train_lora_exp_d.yaml


In [ ]:
!python -m scripts.train --config configs/train_lora_exp_e.yaml


In [ ]:
!python -m scripts.train --config configs/train_lora_exp_f.yaml


In [ ]:
!python -m scripts.train --config configs/train_lora_exp_g.yaml


In [ ]:
!python -m scripts.train --config configs/train_lora_exp_h.yaml


## 8. Template econômico T4 opcional

Use apenas se for treinar em T4 com economia máxima de VRAM. Não misture esses resultados com os experimentos A-H sem documentar a diferença de configuração.


In [ ]:
# !python -m scripts.train --config configs/train_qlora_t4_template.yaml


## 9. Benchmarks finais A-H

Cada célula roda Spider dev e MMLU 150 para um adapter treinado, usando `configs/eval.yaml`.


In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_a


In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_b


In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_c


In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_d


In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_e


In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_f


In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_g


In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_h


## 10. Diagnóstico Spider sem stop sequences

Use estas células quando quiser investigar se `stop_sequences` estão cortando respostas. Os resultados vão para `outputs/diagnostics` e não sobrescrevem os benchmarks oficiais.


In [ ]:
!python -m scripts.evaluate_spider --config configs/eval_spider_nostop.yaml --model_path outputs/base --output_dir outputs/diagnostics/base_spider_nostop


In [ ]:
!python -m scripts.evaluate_spider --config configs/eval_spider_nostop.yaml --model_path outputs/exp_a --output_dir outputs/diagnostics/exp_a_spider_nostop


In [ ]:
!python -m scripts.evaluate_spider --config configs/eval_spider_nostop.yaml --model_path outputs/exp_b --output_dir outputs/diagnostics/exp_b_spider_nostop


In [ ]:
!python -m scripts.evaluate_spider --config configs/eval_spider_nostop.yaml --model_path outputs/exp_c --output_dir outputs/diagnostics/exp_c_spider_nostop


In [ ]:
!python -m scripts.evaluate_spider --config configs/eval_spider_nostop.yaml --model_path outputs/exp_d --output_dir outputs/diagnostics/exp_d_spider_nostop


In [ ]:
!python -m scripts.evaluate_spider --config configs/eval_spider_nostop.yaml --model_path outputs/exp_e --output_dir outputs/diagnostics/exp_e_spider_nostop


In [ ]:
!python -m scripts.evaluate_spider --config configs/eval_spider_nostop.yaml --model_path outputs/exp_f --output_dir outputs/diagnostics/exp_f_spider_nostop


In [ ]:
!python -m scripts.evaluate_spider --config configs/eval_spider_nostop.yaml --model_path outputs/exp_g --output_dir outputs/diagnostics/exp_g_spider_nostop


In [ ]:
!python -m scripts.evaluate_spider --config configs/eval_spider_nostop.yaml --model_path outputs/exp_h --output_dir outputs/diagnostics/exp_h_spider_nostop


## 11. Avaliação opcional de checkpoints

Ajuste o caminho do checkpoint existente antes de executar. Útil para comparar uma época intermediária contra o adapter final.


In [ ]:
# !python -m scripts.evaluate_spider --config configs/eval_spider_nostop.yaml --model_path outputs/exp_c/checkpoint-438 --output_dir outputs/diagnostics/exp_c_ckpt438_spider_nostop
# !python -m scripts.evaluate_mmlu --config configs/eval.yaml --model_path outputs/exp_c/checkpoint-438 --output_dir outputs/diagnostics/exp_c_ckpt438_mmlu


## 12. Smoke mode sem baixar modelos

Use isto apenas para validar o encadeamento dos scripts, não para reportar resultados.


In [ ]:
# !python -m pytest
# !python -m scripts.prepare_mmlu --config configs/eval.yaml --mock --limit_per_category 2
# !python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/base --mock --limit 2
